# Chapter 24 — AI as Builder, Designer, Researcher, and Reviewer

**Book alignment:** Debugging AI From First Principles, Chapter 24

**Question this notebook isolates:** One prompt ("a cached user-profile endpoint with
review") yields four fluent artifacts that fail in four unrelated ways. Does a
role → evidence → failure triage — declare the role, demand *its* evidence before reading
the prose — catch all four where one generic "read it carefully" pass catches none?

In [ ]:
# four artifacts from one prompt; each carries prose, each is defective in its role's way
ARTIFACTS = {
    "builder": dict(prose="Handles all edge cases with a robust TTL cache.",
                    tests_run=False, ttl_refreshes_on_read=True),
    "designer": dict(prose="Parallel fan-out keeps latency flat.",
                     latency_budget_ms=200, hop_latencies_ms=[80, 190, 180]),
    "researcher": dict(prose="Segment-cached invalidation is standard [Smith 2021].",
                       citations=[("Smith 2021", None)]),          # None = no retrieved-byte hash
    "reviewer": dict(prose="Reviewed; approved. Handles edge cases.",
                     lines_changed=47, lines_examined=12),
}

## 1. Generic carefulness: all four read fine

In [ ]:
for role, a in ARTIFACTS.items():
    print(f"[{role:10}] {a['prose']}")
print("\nevery artifact is fluent. a single 'read it over' pass has nothing to catch.")

## 2. Role -> evidence -> failure: demand each role's evidence

In [ ]:
def audit_builder(a):
    # evidence owed: an executed TTL-boundary test
    if not a["tests_run"]:
        return "MISSING evidence: no test run recorded"
    return "TTL test fails (refresh-on-read)" if a["ttl_refreshes_on_read"] else "ok"

def audit_designer(a):
    # evidence owed: line-item arithmetic against the budget
    total = sum(a["hop_latencies_ms"])
    return (f"FAIL: line items {a['hop_latencies_ms']} sum {total}ms > budget {a['latency_budget_ms']}ms"
            if total > a["latency_budget_ms"] else "ok")

def audit_researcher(a):
    # evidence owed: a retrieved byte (hash) per claim
    unresolved = [c for c, h in a["citations"] if h is None]
    return f"UNRESOLVED citations (no source hash): {unresolved}" if unresolved else "ok"

def audit_reviewer(a):
    # evidence owed: coverage of the changed lines
    cov = a["lines_examined"] / a["lines_changed"]
    return f"coverage {a['lines_examined']}/{a['lines_changed']} = {cov:.0%} - approval blessed unexamined paths"

findings = {
    "builder": audit_builder(ARTIFACTS["builder"]),
    "designer": audit_designer(ARTIFACTS["designer"]),
    "researcher": audit_researcher(ARTIFACTS["researcher"]),
    "reviewer": audit_reviewer(ARTIFACTS["reviewer"]),
}
for role, f in findings.items():
    print(f"[{role:10}] {f}")

assert "MISSING" in findings["builder"]
assert "FAIL" in findings["designer"]
assert "UNRESOLVED" in findings["researcher"]
assert "coverage 12/47" in findings["reviewer"]
print("\nfour roles, four distinct evidence types, four distinct failures - one checklist misses three")

## 3. AI-generated evidence is not independent evidence

In [ ]:
model_self_report = "I have tested this thoroughly and it is correct."
independent_check = ARTIFACTS["builder"]["tests_run"]     # False
assert not independent_check
print("the model's own test summary comes from the process that produced the artifact")
print("independence = a check the artifact cannot author: an executed test, arithmetic vs a pinned bound, a retrieved byte")

## What we earned

"AI pair programmer" is four jobs. Declaring the role first and demanding *its* evidence
before reading the prose — an executed test for the builder, line-item arithmetic for the
designer, a retrieved byte per claim for the researcher, coverage of the changed lines for
the reviewer — caught all four defects that fluency hid. A model's own confidence, test
summary, or a second agreeing generation are not independent evidence: independence is a
check the artifact cannot author.

**Notebook 25 / Chapter 25** takes the most common route out of this triage: the intent was
never written down.